# Sequential LT Sampling (Section 5.2)

Evaluates a token-efficient sequential sampling strategy that uses LT signals to decide whether to accept a reasoning trace early or fall back to majority vote (Section 5.2, Figure 5, Table 1).

**Strategy:** Draw model samples sequentially (up to 5). After each sample, compute the LT signal for that trace. If the signal exceeds a learned threshold, accept that answer immediately (spending only the tokens for samples drawn so far). Otherwise continue sampling; if no sample crosses the threshold, fall back to majority vote over all 5.

**Baselines:** MV@5 (majority vote over 5 samples) and Shortest@5 (select the shortest trace).

The notebook covers:
- **Table 1** (Section 5.2): cross-validated accuracy and token savings for individual and combined LT metrics vs. MV@5 and Shortest@5.
- **Appendix G**: cross-dataset threshold transfer — threshold percentile learned on one dataset transferred to another.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import ShuffleSplit

import torch

import sys
sys.path.insert(0, '../src')
from post_utils import create_internals_df

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
PATH = Path('../')  # project root — adjust if running from a different location

## Configuration

In [ ]:
MODELS = ['phi4-reasoning-plus', 'deepseek-r1-qwen14B', 'qwen3-14B']
DATASETS = ['GPQA', 'AIME2025', 'TSP']
AVERAGE_TYPE = 'ntokens_500'

METRICS = [
    'net_change',
    'cumulative_change',
    'aligned_change',
]

MODEL_NAME_MAP = {
    'phi4-reasoning-plus': 'Phi4R+',
    'deepseek-r1-qwen14B': 'R1-D',
    'qwen3-14B': 'Qwen3',
}

LABEL_MAP = {
    'net_change':        'Net Change',
    'cumulative_change': 'Cumulative Change',
    'aligned_change':    'Aligned Change',
    'fused_score':       'LT-Combined',
}

## Helpers

In [ ]:
def compute_majority_vote(internals_df):
    mv_rate = internals_df.groupby('datapoint')['accuracy'].mean()
    maj_vote_acc = (mv_rate > 0.5).mean()
    return maj_vote_acc


def compute_highest_select(internals_df, metric, top='max'):
    if top == 'max':
        idx = internals_df.groupby('datapoint', sort=False)[metric].idxmax()
    elif top == 'min':
        idx = internals_df.groupby('datapoint', sort=False)[metric].idxmin()

    idx = idx.dropna().astype(int)
    if idx.empty:
        return 0.0

    highest_select_df = internals_df.loc[idx]
    return highest_select_df['accuracy'].mean()

## Data loading

In [ ]:
def load_internals_df(model_name, dataset, metrics):
    internals_df, _ = create_internals_df(
        model_name=model_name,
        dataset=dataset,
        path=PATH,
        average_type=AVERAGE_TYPE,
    )
    return internals_df[internals_df['metric_name'].isin(metrics)]

In [ ]:
def compute_fused_metric(internals_df, metrics):
    """Weighted combination of LT signals; weights from Pearson correlation on 10% calibration set."""
    calibration_df = internals_df.sample(frac=0.1, random_state=42)

    # Sign-correct so higher always means more confident
    def sign_correct(df, m):
        s = df[m].astype(float)
        return -s if 'cumulative_change' in m else s

    # Weights from absolute Pearson correlation with accuracy on calibration set
    weights = {}
    for m in metrics:
        s = sign_correct(calibration_df, m)
        corr = abs(s.corr(calibration_df['accuracy'].astype(float)))
        weights[m] = corr if not pd.isna(corr) else 0

    total = sum(weights.values()) or 1
    weights = {k: v / total for k, v in weights.items()}
    print(f"Calibration weights: {weights}")

    # Normalize using calibration statistics, apply to full dataset
    fused = sum(
        (sign_correct(internals_df, m) - sign_correct(calibration_df, m).mean())
        / (sign_correct(calibration_df, m).std(ddof=0) or 1.0) * weights[m]
        for m in metrics
    )
    return fused

## Cross-validated threshold (Section 5.2)

Calibrates a per-metric acceptance threshold using 3-fold cross-validation (30%/70% train/test split). On the training fold, sweep thresholds from the 20th to 99th percentile of incorrect answers' metric values; select the threshold that maximises combined accuracy (accepted samples + MV fallback). Apply that threshold to the test fold to estimate accuracy and token savings.

Columns in the result table:
- `threshold_acc` — accuracy of samples that crossed the threshold
- `full_acc` — overall accuracy (threshold accepts + MV fallback)
- `tokens_used (%)` — fraction of total tokens actually consumed
- `mean_repeats` — average number of samples drawn per question

In [ ]:
def compute_threshold_performance_cv(
        model_name, dataset, metrics, internals_df, fuse_metrics=False, print_result=True,
        n_splits=3, train_size=0.3, test_size=0.7,
        top_k=2, random_state=42, min_incorrect=15,
    ):
    """
    3-fold cross-validation for sequential LT-threshold sampling.

    Calibration (train_size=30%): select threshold from 20th–99th percentiles of
    incorrect answers' metric values; take top_k-median of best thresholds.
    Evaluation (test_size=70%): simulate sequential sampling — accept at first
    sample crossing the threshold, fall back to majority vote otherwise.
    """
    internals_df = (internals_df
        .pivot_table(
            index=['datapoint', 'data_repeat', 'accuracy', 'n_think_tokens'],
            columns='metric_name', values='value',
        ).reset_index())

    mv_acc       = compute_majority_vote(internals_df)
    shortest_acc = compute_highest_select(internals_df, 'n_think_tokens', top='min')

    if print_result:
        print(f'Model: {MODEL_NAME_MAP[model_name]} | Dataset: {dataset}')
        print(f'MV@5 accuracy: {mv_acc:.4f}')
        print(f'Shortest@5 accuracy: {shortest_acc:.4f}')

    if fuse_metrics:
        internals_df['fused_score'] = compute_fused_metric(internals_df, metrics)
        metrics = ['fused_score']

    datapoints = internals_df['datapoint'].unique()
    splitter = ShuffleSplit(
        n_splits=n_splits, train_size=train_size, test_size=test_size, random_state=random_state
    )

    fold_results_all = []
    thresholds_all   = {m: [] for m in metrics}
    sweeps_all       = {m: [] for m in metrics}

    for fold_idx, (train_idx, test_idx) in enumerate(splitter.split(datapoints)):
        train_points = datapoints[train_idx]
        test_points  = datapoints[test_idx]

        train_df = internals_df[internals_df['datapoint'].isin(train_points)].copy()
        test_df  = internals_df[internals_df['datapoint'].isin(test_points)].copy()

        fold_rows = []

        for metric in metrics:
            low_is_good = metric in ['cumulative_change', 'n_think_tokens', 'layerwise_ang']

            false_df = train_df[train_df['accuracy'] == False]

            if len(false_df) < min_incorrect or false_df[metric].notna().sum() == 0:
                best_thr = train_df[metric].median()
            else:
                qs = np.arange(0.20, 1.0, 0.01)
                thr_grid = false_df[metric].quantile(qs).dropna().unique()
                if thr_grid.size == 0:
                    best_thr = train_df[metric].median()
                else:
                    sweep = []
                    for thr in thr_grid:
                        mask = (train_df[metric] <= thr) if low_is_good else (train_df[metric] >= thr)
                        t_df = (train_df[mask]
                                .sort_values(['datapoint', 'data_repeat'])
                                .groupby('datapoint').first().reset_index())
                        t_df_acc = t_df['accuracy'].mean() if not t_df.empty else 0.0

                        th_dp  = set(t_df['datapoint'])
                        not_df = train_df[~train_df['datapoint'].isin(th_dp)]
                        nt_mv  = compute_majority_vote(not_df)
                        n_th, n_nt = len(th_dp), not_df['datapoint'].nunique()
                        overall = ((t_df_acc * n_th) + (nt_mv * n_nt)) / max(1, n_th + n_nt)
                        sweep.append((thr, overall))

                    sweep_df = pd.DataFrame(sweep, columns=['thr', 'acc']).sort_values('acc', ascending=False)
                    best_thr = sweep_df.head(max(1, top_k))['thr'].median()
                    sweeps_all[metric].append(sweep_df)

            full_values    = internals_df[metric].dropna()
            thr_percentile = (full_values <= best_thr).mean() * 100

            thresholds_all[metric].append({
                'threshold': float(best_thr),
                'percentile_full_dataset': float(thr_percentile),
            })

            # Apply threshold to test fold
            mask = (test_df[metric] <= best_thr) if low_is_good else (test_df[metric] >= best_thr)
            t_df = (test_df[mask]
                    .sort_values(['datapoint', 'data_repeat'])
                    .groupby('datapoint').first().reset_index())

            threshold_datapoints = t_df['datapoint'].unique()
            t_df_acc = float(t_df['accuracy'].mean()) if not t_df.empty else 0.0

            # Token and repeat counting
            n_tokens       = 0
            per_dp_repeats = []

            if not t_df.empty:
                tmp = test_df.copy()
                tmp['repeat_idx'] = tmp['data_repeat'].str.extract(r'(\d+)').astype(int)
                for _, row in t_df.iterrows():
                    dp_df = tmp[tmp['datapoint'] == row['datapoint']].sort_values('repeat_idx').reset_index(drop=True)
                    r_idx = dp_df[dp_df['data_repeat'] == row['data_repeat']].index[0]
                    n_tokens += dp_df['n_think_tokens'].iloc[:r_idx + 1].sum()
                    per_dp_repeats.append(r_idx + 1)

            not_threshold_df = test_df[~test_df['datapoint'].isin(threshold_datapoints)]

            if not not_threshold_df.empty:
                per_dp_repeats.extend(
                    not_threshold_df.groupby('datapoint')['data_repeat'].nunique().astype(int).tolist()
                )
                n_tokens_nt = not_threshold_df['n_think_tokens'].sum()
                nt_mv_acc   = compute_majority_vote(not_threshold_df)
                n_th, n_nt  = len(threshold_datapoints), not_threshold_df['datapoint'].nunique()
                full_acc    = ((t_df_acc * n_th) + (nt_mv_acc * n_nt)) / (n_th + n_nt)
            else:
                n_tokens_nt = 0
                full_acc    = t_df_acc

            mean_repeats    = float(np.mean(per_dp_repeats)) if per_dp_repeats else 0.0
            tokens_original = test_df['n_think_tokens'].sum()
            tokens_used_pct = ((n_tokens + n_tokens_nt) / max(1, tokens_original)) * 100

            fold_rows.append([
                metric,
                round(t_df_acc, 3),
                round(full_acc, 3),
                round(tokens_used_pct, 2),
                mean_repeats,
                float(best_thr),
                float(thr_percentile),
            ])

        fold_results_all.append(pd.DataFrame(fold_rows, columns=[
            'metric', 'threshold_acc', 'full_acc', 'tokens_used (%)', 'mean_repeats',
            'best_thr', 'thr_percentile_full_dataset',
        ]))

    results_df = (pd.concat(fold_results_all)
                    .groupby('metric', as_index=False)
                    .mean(numeric_only=True))

    if print_result:
        display(results_df)

    n_tokens_original_full = internals_df['n_think_tokens'].sum()
    return results_df, mv_acc, shortest_acc, n_tokens_original_full, fold_results_all, thresholds_all, sweeps_all

### Table 1 — Sequential LT sampling

Runs the cross-validated threshold sweep for each individual LT metric and for the combined (fused) score across all model × dataset pairs in MODELS × DATASETS. Corresponds to Table 1 in the paper.

In [ ]:
for model_name in MODELS:
    for dataset in DATASETS:
        internals_df = load_internals_df(model_name, dataset, METRICS)

        # Individual LT metrics
        _ = compute_threshold_performance_cv(
            model_name, dataset, METRICS, internals_df,
            fuse_metrics=False, print_result=True,
        )

        # Combined LT metric
        _ = compute_threshold_performance_cv(
            model_name, dataset, METRICS, internals_df,
            fuse_metrics=True, print_result=True,
        )

## Cross-dataset threshold transfer (Appendix G)

Tests whether the threshold percentile learned on one dataset transfers to a different dataset for the same model. For each (model, train_dataset, test_dataset) triple, the CV procedure above yields an optimal threshold percentile; that percentile is then applied to the test dataset's full distribution as a fixed threshold. Reports accuracy and token savings for every leave-one-dataset-out combination.

In [ ]:
def compute_threshold_performance_percentile_single(internals_df, metrics, percentile, print_result=True):
    """Apply a fixed percentile threshold (learned on a different dataset) to internals_df."""
    df = (internals_df
          .pivot_table(
              index=['datapoint', 'data_repeat', 'accuracy', 'n_think_tokens'],
              columns='metric_name', values='value',
          ).reset_index())

    df['fused_score'] = compute_fused_metric(df, metrics)
    metric      = 'fused_score'

    dp_df       = df.sort_values(['datapoint', 'data_repeat']).groupby('datapoint').first().reset_index()
    full_values = dp_df[metric].dropna()
    best_thr    = full_values.quantile(percentile)
    thr_pct     = (full_values <= best_thr).mean() * 100

    if print_result:
        print(f"percentile={percentile*100:.1f}%  threshold={best_thr:.4f}  (full-dataset pct={thr_pct:.2f}%)")

    t_df = (df[df[metric] >= best_thr]
            .sort_values(['datapoint', 'data_repeat'])
            .groupby('datapoint').first().reset_index())

    threshold_datapoints = set(t_df['datapoint'])
    t_df_acc = t_df['accuracy'].mean() if not t_df.empty else 0.0
    not_df   = df[~df['datapoint'].isin(threshold_datapoints)]

    if not not_df.empty:
        nt_mv_acc = compute_majority_vote(not_df)
        n_th, n_nt = len(threshold_datapoints), not_df['datapoint'].nunique()
        full_acc   = ((t_df_acc * n_th) + (nt_mv_acc * n_nt)) / (n_th + n_nt)
    else:
        full_acc = t_df_acc

    n_tokens, per_dp_repeats = 0, []
    if not t_df.empty:
        tmp = df.copy()
        tmp['repeat_idx'] = tmp['data_repeat'].str.extract(r'(\d+)').astype(int)
        for _, row in t_df.iterrows():
            dp_df_tmp = tmp[tmp['datapoint'] == row['datapoint']].sort_values('repeat_idx').reset_index(drop=True)
            r_idx = dp_df_tmp[dp_df_tmp['data_repeat'] == row['data_repeat']].index[0]
            n_tokens += dp_df_tmp['n_think_tokens'].iloc[:r_idx + 1].sum()
            per_dp_repeats.append(r_idx + 1)

    not_grouped = df[~df['datapoint'].isin(threshold_datapoints)]
    if not not_grouped.empty:
        per_dp_repeats.extend(not_grouped.groupby('datapoint')['data_repeat'].nunique().astype(int).tolist())
        n_tokens_nt = not_grouped['n_think_tokens'].sum()
    else:
        n_tokens_nt = 0

    tokens_used_pct = 100 * (n_tokens + n_tokens_nt) / max(1, df['n_think_tokens'].sum())
    mean_repeats    = float(np.mean(per_dp_repeats)) if per_dp_repeats else 0.0

    return pd.DataFrame([[
        metric, round(t_df_acc, 3), round(full_acc, 3),
        round(tokens_used_pct, 2), mean_repeats, float(best_thr), float(thr_pct),
    ]], columns=[
        'metric', 'threshold_acc', 'full_acc', 'tokens_used (%)', 'mean_repeats',
        'best_thr', 'thr_percentile_full_dataset',
    ])

In [ ]:
results = []

for model_name in MODELS:
    print(f"\n{'='*30}\n  MODEL: {model_name}\n{'='*30}")

    for train_dataset in DATASETS:
        print(f"\n▶ Training threshold on: {train_dataset}")

        internals_train = load_internals_df(model_name, train_dataset, METRICS)

        cv_res = compute_threshold_performance_cv(
            model_name=model_name,
            dataset=train_dataset,
            metrics=METRICS,
            internals_df=internals_train,
            fuse_metrics=True,
            print_result=False,
        )

        thr_percentile = cv_res[0]['thr_percentile_full_dataset'].item() / 100.0
        print(f"  Learned percentile = {thr_percentile:.4f}")

        for test_dataset in DATASETS:
            if test_dataset == train_dataset:
                continue
            print(f"  ▶ Testing on: {test_dataset}")
            internals_test = load_internals_df(model_name, test_dataset, METRICS)
            test_df = compute_threshold_performance_percentile_single(
                internals_df=internals_test,
                metrics=METRICS,
                percentile=thr_percentile,
                print_result=False,
            )
            test_df.insert(0, 'test_dataset',  test_dataset)
            test_df.insert(0, 'train_dataset', train_dataset)
            test_df.insert(0, 'model',         model_name)
            results.append(test_df)

results_all = pd.concat(results, ignore_index=True)
print("\n===== FINAL RESULTS =====")
display(results_all)

In [ ]:
results_all.groupby(['model', 'test_dataset']).mean(numeric_only=True).reset_index()